# Audio Preprocessing
---
Handles: noise reduction, silence trimming, normalization, and audio cleanup.

## Importing Audio
---
Load audio file

In [ ]:
import librosa

audio = 'audio.wav'
user_audio, sr = librosa.load(audio, sr=22050)
print(f"Loaded! Duration: {len(user_audio)/sr:.2f}s")

## Silence Trimming
---
Remove silence from beginning and end of audio.

In [ ]:
import numpy as np

def trim_silence(audio, sr, threshold_db=-40):
    print("Trimming silence...")

    # Create spectrogram and convert to dB
    S = librosa.feature.melspectrogram(y=audio, sr=sr)
    S_db = librosa.power_to_db(S, ref=np.max)

    # Get energy per frame
    energy = np.mean(S_db, axis=0)

    # Find frames above threshold (-40 dB soft singing)
    active_frames = energy > threshold_db
    active_indices = np.where(active_frames)[0]

    if len(active_indices) == 0:
        return audio

    # Get start and end frames
    start_frame = active_indices[0]
    end_frame = active_indices[-1]

    # Convert to samples
    start_sample = librosa.frames_to_samples(start_frame, hop_length=512)
    end_sample = librosa.frames_to_samples(end_frame, hop_length=512)

    # Add 100ms buffer for smooth transitions
    buffer = int(0.1 * sr)
    start_sample = max(0, start_sample - buffer)
    end_sample = min(len(audio), end_sample + buffer)

    # Trim and return
    audio_trimmed = audio[start_sample:end_sample]

    print(f"✓ Trimmed silence!")
    print(f"  Original: {len(audio)/sr:.2f}s")
    print(f"  Trimmed:  {len(audio_trimmed)/sr:.2f}s")
    print(f"  Removed:  {(len(audio) - len(audio_trimmed))/sr:.2f}s of silence")

    return audio_trimmed

In [ ]:
## trimming audio through function
audio = trim_silence(audio, sr)

## Noise Reduction
---
Reduce background noise using spectral subtraction.

In [ ]:
# Noise reduction function
def reduce_noise_spectral_subtraction(audio, sr, noise_duration=1.0):
    print("Reducing noise...")

    # Get spectrogram
    S = librosa.feature.melspectrogram(y=audio, sr=sr)

    # Extract noise profile
    noise_frame_count = int(sr * noise_duration / 512)
    noise_sample = S[:, :noise_frame_count]
    noise_profile = np.mean(noise_sample, axis=1, keepdims=True)

    # Subtract noise profile
    S_reduced = S - 2.0 * noise_profile # noise_factor = 2 is the factor which can change aggressiveness of reduced audio
    S_reduced = np.maximum(S_reduced, 0)

    # Convert back to audio
    audio_reduced = librosa.feature.inverse.mel_to_audio(S_reduced, sr=sr)

    # Normalize to original level
    # Spectral noise reduction alters signal amplitude and dynamic range.
    # To ensure consistent feature extraction, especially RMS energy and
    # emotion-related features, I re-scaled the processed signal to match
    # the original peak amplitude.
    max_original = np.max(np.abs(audio))
    max_reduced = np.max(np.abs(audio_reduced))

    if max_reduced > 1e-8:
        audio_reduced *= (max_original / max_reduced) # so that (red_max = org_max)

    print(f"✓ Noise reduction complete!")

    return audio_reduced

In [ ]:
# Apply noise reduction
audio = reduce_noise_spectral_subtraction(audio, sr)

## Audio Normalization
---
Normalize audio to target loudness level (in dB).

In [ ]:
# normalization of audio so that it suits each type of singer (loud or quiet)
def normalize_loudness(audio, target_level_db=-3.0):
    # target = -3 because
    # Digital audio maximum amplitude = 0 dB
    # so if audio crosses 0dB it is clipped but for singing spikes are natural
    # therefore for safe level we normalized according to -3dB

    # Calculate current RMS (loudness)
    rms = np.sqrt(np.mean(audio ** 2))

    if rms == 0:
        return audio

    # Convert to dB
    current_level_db = 20 * np.log10(rms + 1e-10)

    # Calculate gain needed
    gain_db = target_level_db - current_level_db
    gain_linear = 10 ** (gain_db / 20) # dB to linear conversion

    # Apply gain
    audio_normalized = audio * gain_linear

    # Prevent clipping becoz librosa digital limit is (-1 to +1 or 0dB)
    max_val = np.max(np.abs(audio_normalized))
    if max_val > 1.0:
        audio_normalized = audio_normalized / max_val

    print(f"Normalized audio: {current_level_db:.1f} dB -> {target_level_db:.1f} dB")

    return audio_normalized

In [ ]:
# Apply normalization
audio = normalize_loudness(audio)

---
### Preprocessing complete
---

# Segment Matching
---
Handles: auto-matching user audio to reference song, confidence scoring, manual fallback.

## Loading reference Audio
---
Validate audio pair before evaluation.<br>
Checks: minimum length, sufficient pitched content, etc.